# CBUS mini-reproduction: free Kaggle GPU runner (vLLM)  -  UNTESTED
Settings: Accelerator **GPU T4 x2**, **Internet ON**. No API key, no money. Works in an interactive session (do not rely on 'Save & Run All').
The authors' data, the vLLM log and the LLM cache stay in `/tmp` (licensed texts are not saved as output). Results (small CSVs) go to `/kaggle/working/cbus-repro/runs`.
Quota is limited (about 30 GPU-hours per week; check your Kaggle profile), so stop the session when you are done.

## 1. Setup

In [ ]:
import os, subprocess
REPO = 'https://github.com/YOUR-USERNAME/cbus-repro'   # <- your repo (or upload the folder as a Kaggle dataset and copy it)
if not os.path.exists('/kaggle/working/cbus-repro'):
    subprocess.run(['git','clone',REPO,'/kaggle/working/cbus-repro'],check=True)
if not os.path.exists('/tmp/authors_data'):
    subprocess.run(['git','clone','--depth','1','https://github.com/iai-group/emnlp2026-simreading','/tmp/authors_data'],check=True)
os.chdir('/kaggle/working/cbus-repro')
os.environ['SIMREADING_DATA'] = '/tmp/authors_data'
subprocess.run('pip install -q -r requirements.txt',shell=True)
print(os.getcwd())

In [ ]:
!nvidia-smi

## 2. Install vLLM (can take several minutes; if pip complains about torch, restart the kernel once and re-run from the next cell)

In [ ]:
!pip install -q vllm

## 3. Start a local OpenAI-compatible server
Default: **Qwen2.5-7B-Instruct** in fp16, split over both T4s (one T4 has too little memory for 7B fp16). Fallback if that fails: set `MODEL='Qwen/Qwen2.5-3B-Instruct'` and `TP=1`.
If the server dies, read `/tmp/vllm.log` (next cell).

In [ ]:
import subprocess, time, requests
MODEL = 'Qwen/Qwen2.5-7B-Instruct'; TP = 2; NAME = 'local'
log = open('/tmp/vllm.log','w')
env = dict(os.environ, VLLM_ATTENTION_BACKEND='XFORMERS')   # T4 has no FlashAttention-2
proc = subprocess.Popen(['python','-m','vllm.entrypoints.openai.api_server','--model',MODEL,'--served-model-name',NAME,
    '--dtype','half','--tensor-parallel-size',str(TP),'--max-model-len','8192','--gpu-memory-utilization','0.90',
    '--enable-prefix-caching','--port','8000'], stdout=log, stderr=subprocess.STDOUT, env=env)
for i in range(120):
    try:
        if requests.get('http://localhost:8000/v1/models',timeout=2).ok: print('server ready after',i*5,'s'); break
    except Exception: pass
    if proc.poll() is not None: print('server exited; see /tmp/vllm.log'); break
    time.sleep(5)

In [ ]:
!tail -n 30 /tmp/vllm.log

## 4. Smoke test and pilot (20 students). Inspect `meta.json` (drop rate!) and the table before scaling.

In [ ]:
COMMON = '--base-url http://localhost:8000/v1 --model local --max-tokens 400 --workers 32 --cache /tmp/cache.jsonl'
!python scripts/02_run_simulation.py --mode baseline --students 1 --texts-per-student 1 {COMMON} --out-root /tmp/smoke

In [ ]:
for mode in ['baseline','spr','ts']:
    !python scripts/02_run_simulation.py --mode {mode} --students 20 {COMMON} --out-root runs_pilot
!python scripts/03_evaluate.py "runs_pilot/*" --no-authors

In [ ]:
import json,glob
for f in sorted(glob.glob('runs_pilot/*/meta.json')):
    m=json.load(open(f)); print(m['name'], 'dropped_stage1', m['dropped_stage1_calls'], 'dropped_answers', m['dropped_answer_replies'], 'seconds', m['seconds'], m['usage'])
for l in list(open('/tmp/cache.jsonl',encoding='utf-8'))[:6]: print(json.loads(l)['v'][:200].replace('\n',' '),'\n---')

## 5. Main runs (300 students). One cell per condition. Re-running a cell resumes from the cache while the session lives.

In [ ]:
!python scripts/02_run_simulation.py --mode random --students 300
!python scripts/02_run_simulation.py --mode baseline --students 300 {COMMON}

In [ ]:
!python scripts/02_run_simulation.py --mode spr --students 300 {COMMON}

In [ ]:
!python scripts/02_run_simulation.py --mode spr_dropout --students 300 {COMMON}

In [ ]:
!python scripts/02_run_simulation.py --mode ts --students 300 {COMMON}

## 6. Evaluate and export (download `outputs.zip` from the Output panel, then stop the session)

In [ ]:
!python scripts/03_evaluate.py

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/outputs','zip','.','runs'); shutil.make_archive('/kaggle/working/results','zip','.','results')
proc.terminate()
for p in ['/tmp/cache.jsonl','/tmp/authors_data','/tmp/smoke','runs_pilot']:
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else (os.path.exists(p) and os.remove(p))
print('exported; cache and authors data removed')